<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_kvasir_capsule_data_lake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phase 2 - Build the raw medical data lake

Also called data-engineering and data-curation phase as we build a structured raw medical data lake from capsule-endoscopy data before creating visual artifacts or question-answer pairs. The primary dataset is Kvasir-Capsule, which contains capsule-endoscopy videos, labelled images, medically verified finding classes, bounding-box annotations, video identifiers, and frame numbers. The purpose of this phase is to transform the original dataset files into a clean, searchable, and reproducible project data layer.

### 1. Install dependencies and imports

In [ ]:
%pip install -q \
    pandas \
    numpy \
    pyarrow \
    opencv-python-headless \
    scikit-image \
    scikit-learn \
    iterative-stratification \
    tqdm \
    osfclient

In [ ]:
from pathlib import Path
from collections import defaultdict

import os
import re
import json
import random
import hashlib
import warnings
import subprocess
from datetime import datetime, timezone

import cv2
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from pathlib import Path

from skimage.metrics import structural_similarity as ssim

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

### 2. Load declarative configuration and reproducibility

In [ ]:
RUN_ID = datetime.now(ZoneInfo("America/Toronto")).strftime("%Y%m%d_%H%M%S")


CONFIG = {
  "phase": "phase2",
  "experiment_name": "phase2_kvasir_capsule_raw_medical_data_lake",
  "notebook_name": "phase2_kvasir_capsule_data_lake.ipynb",

  "storage_backend": "google_drive",
  "storage_root": "/content/drive/MyDrive/MMVQA_Clinical",

  "dataset_name": "Kvasir-Capsule",
  "dataset_source": "OSF",
  "dataset_osf_project_id": "dv2ag",

  "dataset_validation": {
    "required_metadata_file": "metadata.csv",
    "minimum_video_files": 117,
    "minimum_labelled_images": 47238,
    "video_extensions": [".avi", ".mp4"],
    "image_extensions": [".jpg", ".jpeg", ".png"]
  }

  "expected_labelled_frames": 47238,
  "expected_classes": 14,
  "expected_labelled_videos": 43,
  "expected_unlabelled_videos": 74,
  "expected_total_videos": 117,
  "expected_total_extractable_frames": 4741504,
  "expected_unlabelled_frames": 4694266,

  "dataset_download_enabled": false,

  "raw_data_dir": "data/raw/kvasir_capsule",
  "interim_data_dir": "data/interim/phase2",
  "curated_data_dir": "data/curated/phase2",
  "output_dir": "outputs/phase2",

  "image_extensions": [".png", ".jpg", ".jpeg"],
  "video_extensions": [".avi", ".mp4", ".mkv"],

  "original_capture_fps": 2.0,
  "expected_export_container_fps": 30.0,
  "frame_index_offset_candidates": [-1, 0, 1],
  "frame_alignment_sample_size": 30,

  "split_strategy": "video_level_multilabel_stratified",
  "train_fraction": 0.70,
  "validation_fraction": 0.15,
  "test_fraction": 0.15,

  "segment_max_gap_frames": 1,
  "minimum_verified_segment_frames": 2,

  "domain_adaptation_enabled": true,
  "domain_adaptation_include_fully_unlabelled_videos": true,
  "domain_adaptation_include_train_video_unlabelled_frames": true,
  "domain_adaptation_exclude_validation_videos": true,
  "domain_adaptation_exclude_test_videos": true,
  "domain_adaptation_sampling_seconds": 0.5,
  "domain_adaptation_extract_frames": false,

  "temporal_context_enabled": true,
  "temporal_context_offsets_seconds": [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0],
  "temporal_context_extract_frames": true,
  "temporal_context_image_format": "jpg",
  "temporal_context_jpeg_quality": 95,

  "qc_enabled": true,
  "qc_blur_quantile": 0.05,
  "qc_contrast_quantile": 0.05,
  "qc_brightness_low_quantile": 0.01,
  "qc_brightness_high_quantile": 0.99,

  "clinical_group_map": {
    "Ampulla of Vater": "anatomical_landmark",
    "Ileocecal valve": "anatomical_landmark",
    "Pylorus": "anatomical_landmark",

    "Normal clean mucosa": "normal_mucosa",
    "Reduced mucosal view": "visibility_limitation",

    "Blood - fresh": "bleeding",
    "Blood - hematin": "bleeding",

    "Angiectasia": "vascular_lesion",
    "Erosion": "mucosal_lesion",
    "Erythema": "mucosal_lesion",
    "Ulcer": "mucosal_lesion",

    "Lymphangiectasia": "lymphatic_lesion",
    "Polyp": "protruding_lesion",
    "Foreign body": "foreign_body"
  },

  "seed": 42,
  "run_id": RUN_ID,
  "timezone": "America/Toronto"
}

CONFIG

In [ ]:

# Set random seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"Seed set to: {CONFIG['seed']}")

### 3. Mount Google Drive Storage Backend

In [ ]:
def mount_storage(config):
    """
    Mounts persistent storage when required by the configured backend.

    For Google Colab + Google Drive, this mounts Drive under /content/drive.
    It does not download or copy the dataset.
    """

    storage_backend = config["storage_backend"]

    if storage_backend == "google_drive":

        try:
            from google.colab import drive

            drive.mount(
                "/content/drive",
                force_remount=False,
            )

            print("Google Drive mounted.")

        except ImportError:
            raise RuntimeError(
                "Google Drive backend is configured, "
                "but the notebook is not running in Google Colab."
            )

    elif storage_backend == "local":

        print("Using local storage.")

    else:

        raise ValueError(
            f"Unsupported storage backend: {storage_backend}"
        )


mount_storage(CONFIG)

### 4. Define data paths

In [ ]:
def prepare_phase2_dirs(config):
    """
    Creates the directory structure required for Phase 2.

    Main directories are defined declaratively in CONFIG.
    Relative paths are resolved against storage_root.

    Derived subdirectories such as manifests, splits, reports,
    and temporal frames are created here.

    Side effect:
        Creates folders on persistent storage.

    Returns:
        Dictionary containing resolved Path objects.
    """

    storage_root = Path(
        config["storage_root"]
    )


    def resolve_path(path_value):
        """
        Resolves relative CONFIG paths against storage_root.
        Absolute paths are preserved unchanged.
        """

        path = Path(path_value)

        if path.is_absolute():
            return path

        return storage_root / path


    raw_data_dir = resolve_path(
        config["raw_data_dir"]
    )

    interim_data_dir = resolve_path(
        config["interim_data_dir"]
    )

    curated_data_dir = resolve_path(
        config["curated_data_dir"]
    )

    output_dir = resolve_path(
        config["output_dir"]
    )


    dirs = {
        # Main CONFIG directories
        "raw_data_dir": raw_data_dir,
        "interim_data_dir": interim_data_dir,
        "curated_data_dir": curated_data_dir,
        "output_dir": output_dir,

        # Derived data directories
        "temporal_frames_dir":
            interim_data_dir / "temporal_frames",

        "manifests_dir":
            curated_data_dir / "manifests",

        "splits_dir":
            curated_data_dir / "splits",

        # Derived output directories
        "configs_dir":
            output_dir / "configs",

        "results_dir":
            output_dir / "results",

        "reports_dir":
            output_dir / "reports",
    }


    for directory in dirs.values():

        directory.mkdir(
            parents=True,
            exist_ok=True,
        )


    return dirs


DIRS = prepare_phase2_dirs(CONFIG)

DIRS

### 5. Dataset verification

In [ ]:
def verify_raw_dataset(config, dirs):
    """
    Validates the raw dataset using declarative rules from CONFIG.
    """

    raw_data_dir = dirs["raw_data_dir"]
    validation = config["dataset_validation"]

    if not raw_data_dir.exists():
        return False

    metadata_files = list(
        raw_data_dir.rglob(
            validation["required_metadata_file"]
        )
    )

    video_files = [
        path
        for path in raw_data_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in validation["video_extensions"]
        )
    ]

    image_files = [
        path
        for path in raw_data_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in validation["image_extensions"]
        )
    ]

    valid = (
        len(metadata_files) >= 1
        and len(video_files)
            >= validation["minimum_video_files"]
        and len(image_files)
            >= validation["minimum_labelled_images"]
    )

    return valid

DATASET_AVAILABLE = verify_raw_dataset(
    CONFIG,
    DIRS,
)

DATASET_AVAILABLE

In [ ]:
def download_dataset_if_needed(
    config,
    dirs,
):
    """
    Downloads Kvasir-Capsule only when the raw dataset
    does not satisfy the validation rules defined in CONFIG.

    The download behavior is controlled by CONFIG.

    Side effect:
        Downloads files into dirs["raw_data_dir"] when needed.
    """

    dataset_available = verify_raw_dataset(
        config,
        dirs,
    )

    if dataset_available:
        print(
            "Dataset already available and valid. "
            "Skipping download."
        )
        return

    if not config["dataset_download_enabled"]:
        raise RuntimeError(
            "Kvasir-Capsule is missing or incomplete, "
            "and automatic download is disabled."
        )

    raw_data_dir = dirs["raw_data_dir"]

    raw_data_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    command = [
        "osf",
        "-p",
        config["dataset_osf_project_id"],
        "clone",
        str(raw_data_dir),
    ]

    print(
        "Downloading Kvasir-Capsule to persistent storage..."
    )

    subprocess.run(
        command,
        check=True,
    )

    # Validate again after download.
    dataset_available = verify_raw_dataset(
        config,
        dirs,
    )

    if not dataset_available:
        raise RuntimeError(
            "Dataset download completed, but the dataset "
            "does not satisfy the validation rules defined in CONFIG."
        )

    print(
        "Dataset download completed and validation passed."
    )


download_dataset_if_needed(
    CONFIG,
    DIRS,
)